# Deep Learning Model (PyTorch MLP)
This notebook trains a feed-forward Multilayer Perceptron (MLP) to satisfy the 'Deep Learning Plus AI' module requirement. We will then compare its performance against the Ridge and LightGBM baselines.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

# Add src to path
sys.path.append(os.path.abspath('../src'))
from data_loader import get_all_well_ids, load_well_data
from features import extract_features_for_well

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.backends.mps.is_available():
    device = torch.device('mps')
print(f"Using device: {device}")

Using device: mps


In [2]:
DATA_DIR = '../data'
well_ids = get_all_well_ids(DATA_DIR, split='train')

# Using 100 wells for demonstration
num_wells_to_use = 100
selected_wells = well_ids[:num_wells_to_use]

all_features = []
for wid in selected_wells:
    horz_df, type_df = load_well_data(wid, DATA_DIR, split='train')
    df_feats = extract_features_for_well(wid, horz_df, type_df, is_train=True)
    if len(df_feats) > 0:
        all_features.append(df_feats)

dataset = pd.concat(all_features, ignore_index=True)
dataset['target_dTVT'] = dataset['TVT'] - dataset['ps_TVT']

features = ['delta_Z', 'delta_MD', 'dist_XY', 'GR', 'inclination_proxy']

# Neural networks require no NaNs and properly scaled inputs
X = dataset[features].fillna(0).values
y = dataset['target_dTVT'].values.reshape(-1, 1)
groups = dataset['well_id'].values

In [3]:
class WellboreMLP(nn.Module):
    def __init__(self, input_dim):
        super(WellboreMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        return self.net(x)

In [4]:
gkf = GroupKFold(n_splits=5)
rmse_scores = []
epochs = 20
batch_size = 256

print("Starting PyTorch GroupKFold CV...")
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups)):
    # Split data
    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]
    
    # Scale data (fit on train, transform both)
    scaler_x = StandardScaler()
    X_train_scaled = scaler_x.fit_transform(X_train)
    X_val_scaled = scaler_x.transform(X_val)
    
    # Convert to Tensors
    X_tr_t = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
    y_tr_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32).to(device)
    
    train_ds = TensorDataset(X_tr_t, y_tr_t)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    
    # Initialize Model
    model = WellboreMLP(input_dim=len(features)).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.005)
    criterion = nn.MSELoss()
    
    # Training loop
    model.train()
    for epoch in range(epochs):
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            preds = model(batch_x)
            loss = criterion(preds, batch_y)
            loss.backward()
            optimizer.step()
            
    # Evaluation
    model.eval()
    with torch.no_grad():
        preds_val = model(X_val_t).cpu().numpy()
        
    # Calculate absolute TVT prediction to get actual RMSE in feet
    ps_TVT_val = dataset.iloc[val_idx]['ps_TVT'].values.reshape(-1, 1)
    pred_abs_TVT = ps_TVT_val + preds_val
    true_abs_TVT = dataset.iloc[val_idx]['TVT'].values.reshape(-1, 1)
    
    score = root_mean_squared_error(true_abs_TVT, pred_abs_TVT)
    rmse_scores.append(score)
    print(f"Fold {fold+1} RMSE: {score:.2f} feet")

print(f"\nAverage CV RMSE (MLP): {np.mean(rmse_scores):.2f} ± {np.std(rmse_scores):.2f} feet")

Starting PyTorch GroupKFold CV...
Fold 1 RMSE: 16.22 feet
Fold 2 RMSE: 16.48 feet
Fold 3 RMSE: 19.96 feet


KeyboardInterrupt: 

## Final Model Comparison
Update the lists below with the actual output values from your previous notebooks to generate the final comparison chart for your report.

In [ ]:
# Example placeholders - replace these with your actual output averages
ridge_rmse = 15.50  # From 02_Baseline_Model.ipynb
lgbm_rmse = 9.20    # From 03_Ensemble_Model.ipynb
mlp_rmse = np.mean(rmse_scores) # From this notebook

models = ['Ridge Regression (Baseline)', 'LightGBM (Ensemble)', 'PyTorch MLP (Deep Learning)']
scores = [ridge_rmse, lgbm_rmse, mlp_rmse]

plt.figure(figsize=(10, 6))
sns.barplot(x=models, y=scores, palette='viridis')
plt.title('Model Comparison: Cross-Validation RMSE (Lower is Better)')
plt.ylabel('RMSE (feet)')
plt.xlabel('Model')

for i, v in enumerate(scores):
    plt.text(i, v + 0.5, f"{v:.2f}", ha='center', va='bottom', fontweight='bold')

plt.show()